# Laboratorio di Analisi Numerica
## Lezione 5

Cecilia Pagliantini <cecilia.pagliantini@unipi.it>  
Leonardo Robol <leonardo.robol@unipi.it>  
31 ottobre 2024

*Conversione della dispensa «Laboratorio 05.pdf»; data mantenuta dall'originale.*

> **Quantità di esercizi:** in questa dispensa ci sono *più esercizi* di quanti uno studente medio riesca a farne durante una lezione di laboratorio, specialmente tenendo conto anche degli esercizi facoltativi. Questo perché è pensata per “tenere impegnati” per tutta la lezione anche quegli studenti che già hanno un solido background di programmazione.
>
> Quindi fate gli esercizi che riuscite, partendo da quelli *non* segnati come facoltativi, e non preoccupatevi se non li finite tutti!

Questa lezione usa `LinearAlgebra`, incluso nella libreria standard di Julia. Eseguite le celle dall'alto verso il basso e completate gli esercizi negli spazi predisposti. Le chiamate alle funzioni da scrivere sono commentate: togliete `#` dopo averle definite.

Le funzioni per i sistemi triangolari sono riportate qui, quindi non occorre eseguire un altro notebook. In questa lezione lavoriamo con matrici e vettori reali.

In [ ]:
using LinearAlgebra

## 1. Fattorizzazione LU

Ricordate come funziona il passo $k$-esimo di fattorizzazione LU. Nello schema seguente $k=3$:

$$
L^{(k)}=\begin{pmatrix}
1&0&0&0&0&0\\
0&1&0&0&0&0\\
0&0&1&0&0&0\\
0&0&\color{red}{\ast}&1&0&0\\
0&0&\color{red}{\ast}&0&1&0\\
0&0&\color{red}{\ast}&0&0&1
\end{pmatrix},\qquad
A^{(k)}=\begin{pmatrix}
\ast&\ast&\ast&\ast&\ast&\ast\\
0&\ast&\ast&\ast&\ast&\ast\\
0&0&\color{red}{A^{(k)}_{33}}&\color{red}{\ast}&\color{red}{\ast}&\color{red}{\ast}\\
0&0&\color{red}{A^{(k)}_{43}}&\color{red}{\ast}&\color{red}{\ast}&\color{red}{\ast}\\
0&0&\color{red}{A^{(k)}_{53}}&\color{red}{\ast}&\color{red}{\ast}&\color{red}{\ast}\\
0&0&\color{red}{A^{(k)}_{63}}&\color{red}{\ast}&\color{red}{\ast}&\color{red}{\ast}
\end{pmatrix}.
$$

Dovete moltiplicare $A^{(k)}$ per una matrice $L^{(k)}$ della forma indicata in modo che gli elementi in posizione $(k+1,k),(k+2,k),\dots,(n,k)$ diventino zero. La matrice $L$ è il prodotto delle matrici $(L^{(k)})^{-1}$, che non va veramente calcolato ma, grazie alle proprietà delle matrici elementari di Gauss, si può determinare senza sforzi ulteriori con la relazione

$$
L_{ij}=\frac{A_{ij}^{(j)}}{A_{jj}^{(j)}},\qquad\forall i\geq j.
$$

### Esercizio 1

Scrivete una funzione `my_lu(A)` che calcoli la fattorizzazione LU di una matrice `A`, senza pivoting, secondo il procedimento appena descritto.

**Julia:** per restituire i due fattori usate `return L, U`; la chiamata si scrive `L, U = my_lu(A)`. Se modificate la matrice durante l'eliminazione, lavorate su una copia in virgola mobile, ottenuta con `float.(A)`: l'assegnazione `U = A` da sola non crea una copia e le modifiche a `U` cambierebbero anche `A`.

In [ ]:
# Scrivete qui la funzione my_lu(A)

Testiamo sulla matrice seguente:

In [ ]:
M = 4 * Matrix{Float64}(I, 5, 5) + ones(5, 5)

In [ ]:
# L, U = my_lu(M)
# L * U - M

Nella dispensa originale la differenza contiene zeri e piccoli valori come `-8.88178419700125e-16` e `2.22044604925031e-16`. I valori esatti possono cambiare con l'implementazione. Perché non avete degli zeri esatti?

### Esercizio 2

Scrivete una funzione `sys_solve(A, b)` che risolva un sistema lineare generico utilizzando la fattorizzazione LU e i metodi implementati in `inf_solve` e `sup_solve`. Potete utilizzare le funzioni della volta scorsa per risolvere sistemi triangolari, oppure le seguenti.

**Julia:** `x = float.(b)` crea un nuovo vettore in virgola mobile. Scrivere `x = b`, come nel codice MATLAB originale, farebbe modificare anche il termine noto del chiamante; inoltre un vettore di interi non potrebbe contenere i risultati non interi delle divisioni.

In [ ]:
function inf_solve(L, b)
    # risolve un sistema con L triangolare inferiore
    n = size(L, 1)
    x = float.(b)  # copia: x è un "vettore di accumulatori"
    for i = 1:n
        x[i] = x[i] / L[i, i]
        x[i+1:n] = x[i+1:n] - L[i+1:n, i] * x[i]
    end
    return x
end

In [ ]:
function sup_solve(U, b)
    # risolve un sistema con U triangolare superiore
    n = size(U, 1)
    x = float.(b)  # copia: x è un "vettore di accumulatori"
    for i = n:-1:1
        x[i] = x[i] / U[i, i]
        x[1:i-1] = x[1:i-1] - U[1:i-1, i] * x[i]
    end
    return x
end

In [ ]:
# Scrivete qui la funzione sys_solve(A, b)

### Esercizio 3 (facoltativo)

Capite come funziona questa versione di `inf_solve` e `sup_solve`. Notate che i calcoli che esegue non sono identici alla versione della scorsa lezione! Ricordate che la formula è

$$
x_i=\frac{b_i-\sum_{j=1}^{i-1}L_{ij}x_j}{L_{ii}},\qquad i=1,\dots,n.
$$

In [ ]:
# Inserite qui gli esperimenti per confrontare le due versioni

## 2. Esperimenti numerici

### Esercizio 4

Testare i seguenti metodi di soluzione di un sistema lineare $Ax=b$:

- La funzione `sys_solve` appena scritta.
- Il comando `x = inv(A)*b`, che calcola la matrice inversa e la moltiplica per `b`.
- Il comando `x = A \ b`: l'operatore `\` (barra rovesciata o *backslash*) serve proprio per risolvere sistemi lineari. Per una matrice densa quadrata generica Julia usa la fattorizzazione LU con pivoting parziale, più stabile della semplice eliminazione senza pivoting; per matrici con strutture particolari può scegliere altri metodi.

Per testarli, utilizzate le seguenti sei matrici. Le celle qui sotto ne traducono la costruzione in Julia; il confronto dei metodi è da completare.

La matrice `M1` è a diagonale dominante. `M2` ha elementi casuali e può essere abbastanza mal condizionata! Potete controllare il condizionamento con `cond(M2)`.

**Julia:** `ones(10, 10)` e `rand(10, 10)` producono matrici $10\times 10$; con un solo argomento produrrebbero vettori.

In [ ]:
M1 = 9 * Matrix{Float64}(I, 10, 10) + ones(10, 10)
M2 = rand(10, 10)

La matrice `M3` ha una riga quasi tutta di zeri, che rende la sottomatrice principale $9\times9$ singolare e impedisce la fattorizzazione LU senza scambi di righe di questa matrice.

**Julia:** usiamo `copy` per preservare `M1` e `.=` per assegnare uno scalare a tutti gli elementi della porzione selezionata.

In [ ]:
M3 = copy(M1)
M3[9, 1:9] .= 0

Nella matrice `M4`, i primi nove elementi della nona riga sono ognuno la somma degli otto elementi che stanno direttamente sopra di esso. Quindi la sottomatrice principale $9\times9$ è singolare (ma lavorando con i numeri floating point…).

**Julia:** `sum(B; dims=1)` somma lungo le righe e mantiene una matrice con una sola riga. `vec` la converte in un vettore, come quello selezionato da `M4[9, 1:9]`. Senza `dims`, `sum(B)` sommerebbe tutti gli elementi.

In [ ]:
M4 = copy(M2)
M4[9, 1:9] = vec(sum(M4[1:8, 1:9]; dims=1))

Gli elementi modificati sono evidenziati in rosso:

$$
\begin{smallmatrix}
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast\\
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast\\
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast\\
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast\\
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast\\
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast\\
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast\\
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast\\
\color{red}{\ast} & \color{red}{\ast} & \color{red}{\ast} & \color{red}{\ast} & \color{red}{\ast} & \color{red}{\ast} & \color{red}{\ast} & \color{red}{\ast} & \color{red}{\ast} & \ast\\
\ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast & \ast
\end{smallmatrix}.
$$

Nella matrice `M5`, l'ultima riga è la somma delle nove precedenti, quindi la matrice è singolare (ma lavorando con i numeri floating point…).

In [ ]:
M5 = copy(M2)
M5[10, 1:10] = vec(sum(M5[1:9, 1:10]; dims=1))

La matrice `M6` è la matrice di Hilbert di ordine 10. In Julia la costruiamo con una comprensione, al posto del comando MATLAB `hilb(10)`. Riuscite a spiegare il numero di cifre significative ottenute in base al suo numero di condizionamento, che potete calcolare con `cond(M6)`?

In [ ]:
M6 = [1 / (i + j - 1) for i = 1:10, j = 1:10]

Ponete `v = collect(1.0:10.0)`, vettore contenente i numeri da 1 a 10 in ordine; per ognuna di queste matrici calcolate $b_i=M_i v$ (per $i=1,\dots,6$), e risolvete il sistema $M_i x=b_i$. Il vettore `v` è una soluzione esatta del sistema costruito in aritmetica esatta (non necessariamente l'unica per le matrici singolari); di quanto si discostano le soluzioni calcolate?

`v` è un vettore monodimensionale e non va trasposto per usarlo in `Mi*v`. Nella costruzione numerica del termine noto intervengono anche gli arrotondamenti.

Alcuni metodi possono segnalare un errore su queste matrici: annotate anche questo esito e continuate il confronto sulle altre. Potete eseguire i casi separatamente o usare `try ... catch` per non interrompere l'intero esperimento.

In [ ]:
v = collect(1.0:10.0)
# Scrivete qui il confronto fra sys_solve(A, b), inv(A)*b e A \ b
# per M1, M2, M3, M4, M5 e M6

### Esercizio 5 (facoltativo)

Guardate la fattorizzazione LU **senza pivoting**, calcolata con `my_lu(M4)`. `U[9, 9]` è molto piccolo; perché? `U[10, 10]` è molto grande; perché?

Il risultato dipende dagli arrotondamenti: se il pivot diventa esattamente nullo, la routine può invece interrompersi. La fattorizzazione standard `lu(M4)` usa scambi di righe e non è quella a cui si riferisce questo esperimento.

In [ ]:
# Inserite qui gli esperimenti sui fattori di M4

### Esercizio 6

Prendete

$$
L=\begin{bmatrix}1&0&0&0\\1&1&0&0\\1&0&1&0\\1&0&0&1\end{bmatrix},\qquad
U=\begin{bmatrix}1&1&1&1\\0&0&0&0\\0&0&0&0\\0&0&0&0\end{bmatrix}.
$$

Quanto vale la matrice `A = L*U`? La matrice $A$ soddisfa le ipotesi sufficienti del teorema visto a lezione per l'esistenza della fattorizzazione LU? Che risultato ritorna la vostra routine `my_lu(A)` su questa matrice?

In [ ]:
L = [1.0 0 0 0; 1 1 0 0; 1 0 1 0; 1 0 0 1]
U = [1.0 1 1 1; 0 0 0 0; 0 0 0 0; 0 0 0 0]
# Completate qui l'esercizio 6

## 3. Facoltativo: eliminazione di Gauss “senza L”

Si possono riorganizzare i calcoli in modo che il calcolo di $L$ venga effettuato implicitamente; questo probabilmente è equivalente a come avete visto l'eliminazione di Gauss lo scorso anno ad algebra lineare. Lavoriamo sulle equazioni anziché sulle matrici:

$$
\begin{aligned}
a_{11}x_1+a_{12}x_2+a_{13}x_3&=b_1,\\
a_{21}x_1+a_{22}x_2+a_{23}x_3&=b_2,\\
a_{31}x_1+a_{32}x_2+a_{33}x_3&=b_3.
\end{aligned}
$$

Cerchiamo di eliminare $x_1$ dalla seconda e terza equazione:

$$
\begin{aligned}
a_{11}x_1+a_{12}x_2+a_{13}x_3&=b_1,\\
a_{21}x_1+a_{22}x_2+a_{23}x_3-\frac{a_{21}}{a_{11}}(a_{11}x_1+a_{12}x_2+a_{13}x_3)&=b_2-\frac{a_{21}}{a_{11}}b_1,\\
a_{31}x_1+a_{32}x_2+a_{33}x_3-\frac{a_{31}}{a_{11}}(a_{11}x_1+a_{12}x_2+a_{13}x_3)&=b_3-\frac{a_{31}}{a_{11}}b_1.
\end{aligned}
$$

Quindi reiteriamo, fino a trasformare $A$ in una matrice triangolare.

Prendiamo come riferimento per l'analisi il primo passo. Come vengono modificati quindi la matrice dei coefficienti $A$ e il termine noto $b$ che stiamo tenendo in memoria? Sulla matrice $A$ facciamo le stesse operazioni usate nel calcolo della fattorizzazione LU. La quantità che dobbiamo sottrarre ad $A$ è una matrice di rango 1:

$$
A_{2:3,1:3}\leftarrow A_{2:3,1:3}-
\begin{bmatrix}a_{21}/a_{11}\\a_{31}/a_{11}\end{bmatrix}
\begin{bmatrix}a_{11}&a_{12}&a_{13}\end{bmatrix}.
$$

In più, dobbiamo effettuare un calcolo simile anche su $b$:

$$
b_{2:3}\leftarrow b_{2:3}-\begin{bmatrix}a_{21}/a_{11}\\a_{31}/a_{11}\end{bmatrix}b_1.
$$

In Julia il primo passo su una matrice $3\times3$ e sul suo termine noto si può scrivere così:

```julia
m = A[2:3, 1] / A[1, 1]
A[2:3, 1:3] = A[2:3, 1:3] - m * transpose(A[1, 1:3])
b[2:3] = b[2:3] - m * b[1]
```

**Julia:** `A[1, 1:3]` è un vettore; `transpose` lo rende una riga per ottenere il prodotto esterno. Calcolate i moltiplicatori `m` prima di modificare la prima colonna di `A`.

Possiamo anche interpretare i calcoli effettuati in termini di prodotti di matrici: nel calcolo della fattorizzazione LU ottenevamo $L^{-1}$ come prodotto di matrici parziali $L_{n-1}L_{n-2}\cdots L_1$, ognuna della forma

$$
\begin{bmatrix}I&&\\&1&\\&v&I\end{bmatrix}.
$$

Ora quello che facciamo è effettuare subito il prodotto con la matrice parziale

$$
L_1=\begin{bmatrix}
1&0&0\\-a_{21}/a_{11}&1&0\\-a_{31}/a_{11}&0&1
\end{bmatrix},
$$

in modo da trasformare il sistema nel sistema equivalente (che ha la stessa soluzione)

$$L_1Ax=L_1b.$$

Al termine degli $(n-1)$ passi, abbiamo trasformato il sistema in un sistema equivalente ma con matrice dei coefficienti triangolare, e questo lo sappiamo risolvere.

### Esercizio 7 (facoltativo)

Scrivere una funzione `sys_solve2(A, b)` che risolva un sistema lineare con il metodo qui sopra.

Lavorate su copie in virgola mobile di `A` e `b` per non modificare gli argomenti del chiamante.

In [ ]:
# Scrivete qui la funzione sys_solve2(A, b)

## 4. Facoltativo: soluzione di un sistema lineare con fattorizzazione QR implicita (Householder)

Una trasformazione di Householder è una matrice ortogonale della forma

$$H_v=I-\frac{2vv^T}{\lVert v\rVert_2^2},\qquad v\neq0.$$

Fissato $x$, esiste una scelta di $v$ che determina una $H_v$ che “porta” $x$ in un multiplo di $e_1$, ed è

$$v=x\pm\lVert x\rVert_2e_1.$$

In particolare, a meno di riorganizzare le operazioni, $v$ è uguale a $x$ tranne per il primo elemento, che vale

$$v_1=x_1-\lVert x\rVert_2=\frac{-(x_2^2+\dots+x_n^2)}{x_1+\lVert x\rVert_2}$$

oppure

$$v_1=x_1+\lVert x\rVert_2=\frac{-(x_2^2+\dots+x_n^2)}{x_1-\lVert x\rVert_2}.$$

Le forme razionalizzate si intendono quando il denominatore è non nullo. In una di queste, il denominatore soffre di errori di cancellazione; quale? Dipende dal segno di $x_1$ e da come scegliamo di calcolare $v$.

**Julia:** `norm(x)` è la norma euclidea del vettore e `v * transpose(v)` è il prodotto esterno $vv^T$. Se volete costruire esplicitamente $H_v$ per verificarne l'effetto, l'identità è `Matrix{Float64}(I, length(v), length(v))`. Lavoriamo con vettori reali, per cui trasposta e aggiunta coincidono.

### Esercizio 8 (facoltativo)

Scrivete una funzione `householder_vector(x)` che calcoli `v` con il segno giusto.

Se modificate la prima componente di `x` per costruire `v`, create prima una copia. Attenzione anche al caso `x[1] == 0`: in Julia `sign(0.0)` vale zero, mentre qui la scelta del segno deve essere fra $+1$ e $-1$. Se il vettore da trasformare è nullo, non occorre eliminarne alcuna componente: gestite questo caso senza dividere per la norma di un vettore nullo.

In [ ]:
# Scrivete qui la funzione householder_vector(x)

### Esercizio 9 (facoltativo)

Scrivete una funzione `qr_solve(A, b)` che risolva un sistema lineare in modo simile a `sys_solve2` ma attraverso la fattorizzazione QR: per esempio, al primo passo, invece di trovare una matrice triangolare $L_1$ che mandi la prima colonna di $A$ in un multiplo di $e_1$ (come facevate in `sys_solve2`), dovete trovare una matrice di Householder $H_1$ che faccia lo stesso lavoro e applicarla sia ad $A$ che a $b$ per ottenere un sistema equivalente:

$$H_1Ax=H_1b.$$

Anche qui, lavorate su copie di `A` e `b`.

In [ ]:
# Scrivete qui la funzione qr_solve(A, b)

### Esercizio 10 (facoltativo)

Cosa succede se usate il segno sbagliato nella funzione `householder_vector`, o se usate sempre lo stesso segno? Testare provando ad applicare la trasformazione di Householder in cui $\lVert x\rVert_2\approx|x_1|$.

In [ ]:
# Inserite qui gli esperimenti sulla scelta del segno

Riferimenti per gli adattamenti: [differenze fra Julia e MATLAB](https://docs.julialang.org/en/v1/manual/noteworthy-differences/) e [algebra lineare in Julia](https://docs.julialang.org/en/v1/stdlib/LinearAlgebra/).